# Importing necessary libraries

In [0]:
import datetime
import os
import re
from dotenv import load_dotenv
from pyspark.sql.functions import (
    col, lit, coalesce, regexp_extract,
    regexp_replace, when, expr
)

# Loading environment variables

In [0]:
load_dotenv(".env", override=True)
LOG_PATH = os.getenv("LOG_PATH")
INPUT_PATH = os.getenv("INPUT_PATH")
OUTPUT_PATH = os.getenv("OUTPUT_PATH")

# 1. Logger Class

In [0]:
class DualLogger:
    """Logs to console + timestamped file with structured messages."""

    def __init__(self, base_path=LOG_PATH):
        os.makedirs(base_path, exist_ok=True)
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        self.log_filename = os.path.join(base_path, f"pipeline_run_{timestamp}.log")

        try:
            self.file = open(self.log_filename, "a", encoding="utf-8")
        except Exception as e:
            print(f"Logger init failed: {e}")
            self.file = None

        self.log("===== PIPELINE SESSION STARTED =====")

    def log(self, message):
    # Convert any path-like string in the message to filename only
    # If entire message itself is a path:
        if isinstance(message, str) and ("/" in message or "\\" in message):
            message = message.replace("\\", "/")

            # Split message into words and strip paths down to basenames
            parts = []
            for part in message.split():
                if "/" in part:
                    parts.append(os.path.basename(part))
                else:
                    parts.append(part)

            message = " ".join(parts)

        timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        msg = f"[{timestamp}] {message}"

        print(msg)

        if self.file:
            try:
                self.file.write(msg + "\n")
                self.file.flush()
            except Exception:
                pass

    def log_df(self, df, stage):
        try:
            count = df.count()
            self.log(f"{stage} → row_count = {count}, columns = {len(df.columns)}")
        except Exception as e:
            self.log(f"{stage} → failed to fetch stats: {e}")

    def close(self):
        self.log("===== PIPELINE SESSION ENDED =====")
        if self.file:
            self.file.close()

# Helper functions

# 1. Data Ingestion
- Read all CSV files using PySpark

- Ensure dynamic file handling (no hardcoding of file names)

- Add a new column: category_name

## (a) - `process_file(file_path)`
Loads and performs initial processing on a single CSV file.

In [0]:
def process_file(file_path):
    logger.log(f"Loading file: {file_path}")

    df = spark.read.csv(file_path, header=True, inferSchema=False)

    logger.log_df(df, "After Load")

    df = standardize_columns(df)
    logger.log("Columns standardized")

    df = add_category(df, file_path)
    logger.log("Category column added")

    return df

## (b) - `extract_category(file_path)`
Extracting the product category from the file name of the csv file

In [0]:
def extract_category(file_path):
    filename = os.path.basename(file_path)
    name = filename.split(".")[0]
    parts = name.split("-")

    if len(parts) < 2:
        return "Unknown"

    return parts[-2].replace("_", " ").title()

## (c) - `add_category(df, file_path)`
Adds the `category_name` column to the DataFrame.

In [0]:
def add_category(df, file_path):
    category = extract_category(file_path)
    return df.withColumn("category_name", lit(category))

## (d) - `load_and_combine(files)`
Loads and combines multiple category DataFrames into one dataset.

In [0]:
def load_and_combine(files):
    combined_df = None

    for i, file in enumerate(files):
        df = process_file(file)

        if combined_df is None:
            combined_df = df
        else:
            combined_df = combined_df.unionByName(df, allowMissingColumns=True)

        logger.log(f"Files processed: {i+1}/{len(files)}")

    logger.log_df(combined_df, "After Combine")

    return combined_df

# 2. Data Cleaning
- Standardize column names

- Handle missing/null values

- Handle columns that exist in some files but not others

- Remove unnecessary columns (if identified)

## (e) - `standardize_columns(df)`
Standardizes the column names.

In [0]:
def standardize_columns(df):
    return df.toDF(*[
        re.sub(r'[^a-zA-Z0-9_]', '_', c) for c in df.columns
    ])

## (f) - `ensure_column(df, col_name, dtype="string")`
Ensures a column exists, adding it if missing.

In [0]:
def ensure_column(df, col_name, dtype="string"):
    if col_name not in df.columns:
        return df.withColumn(col_name, lit(None).cast(dtype))
    return df

## (g) - `select_required_columns(df)`
Selects and retains only the required columns for the final output dataset.

In [0]:
def select_required_columns(df):
    selected_cols = [
        "product_title",
        "category_name",
        "rank",
        "rank_category",
        "price_usd",
        "pct_discount",
        "qty_sold",
        "color_count"
    ]

    return df.select(
        *[c for c in selected_cols if c in df.columns]
    )

# 3. Data Transformation
- Convert string values to appropriate data types:

  - Remove $ from price

  - Remove % from discount

  - Convert "k" values (e.g., 5k → 5000)

- Create standardized columns:

  - price_usd

  - pct_discount

  - qty_sold

- Extract numeric rank from values like #1, #2

## (h) - `merge_title_columns(df)`
Combines the value of two columns to form the `product_title`

In [0]:
def merge_title_columns(df):
    possible_cols = ["goods_title_link__jump", "goods_title_link"]
    existing = [col(c) for c in possible_cols if c in df.columns]

    if existing:
        return df.withColumn("product_title", coalesce(*existing))
    else:
        return df.withColumn("product_title", lit(None).cast("string"))

## (i) - `transform_rank_columns(df)`
Extracts and transforms rank-related columns.

In [0]:
def transform_rank_columns(df):
    if "rank_title" in df.columns:
        val = regexp_extract(col("rank_title"), r'(\d+)', 1)
        df = df.withColumn(
            "rank",
            when(val != '', val.cast("int")).otherwise(None).cast("int")
        )
    else:
        df = df.withColumn("rank", lit(None).cast("int"))

    if "rank_sub" in df.columns:
        df = df.withColumn(
            "rank_category",
            regexp_replace(col("rank_sub"), r'(?i)^(top|rated|in|Top|Rated|In)\s+', '')
        )
    else:
        df = df.withColumn("rank_category", lit(None).cast("string"))

    return df

## (j) - `transform_price(df)`
Cleans and converts raw price values into `price_usd`.

In [0]:
def transform_price(df):
    if "price" in df.columns:
        val = regexp_extract(col("price"), r'(\d+\.?\d*)', 1)
        return df.withColumn(
            "price_usd",
            when(val != '', val.cast("float")).otherwise(None)
        )
    return df.withColumn("price_usd", lit(None).cast("float"))

## (k) - `transform_quantity_sold(df)`
Transforms quantity sold values and converts `k` notation into numeric values.

In [0]:
def transform_quantity_sold(df):
    if "selling_proposition" not in df.columns:
        return df.withColumn("qty_sold", lit(None).cast("int"))

    val = regexp_extract(col("selling_proposition"), r'(\d+\.?\d*)', 1)

    df = df.withColumn(
        "base_value",
        when(val != '', val.cast("float")).otherwise(None)
    )

    df = df.withColumn(
        "qty_sold",
        when(
            col("selling_proposition").rlike("(?i)k") & col("base_value").isNotNull(),
            col("base_value") * 1000
        ).when(
            col("base_value").isNotNull(),
            col("base_value")
        ).otherwise(None)
    )

    return df.withColumn(
        "qty_sold",
        when(col("qty_sold").isNotNull(), col("qty_sold").cast("int")).otherwise(None)
    ).drop("base_value")

## (l) - `transform_discount(df)`
Cleans and converts discount values into `pct_discount`.

In [0]:
def transform_discount(df):
    if "discount" in df.columns:
        val = regexp_extract(col("discount"), r'(\d+\.?\d*)', 1)
        return df.withColumn(
            "pct_discount",
            when(val != '', val.cast("float")).otherwise(None)
        )
    return df.withColumn("pct_discount", lit(None).cast("float"))

## (m) - `apply_transformations(df)`
Applies all transformation steps to prepare the final structured dataset.

In [0]:
def apply_transformations(df):
    logger.log("Starting transformations")

    df = merge_title_columns(df)
    logger.log("Merged title columns")

    df = ensure_column(df, "color_count", "int")
    logger.log("Ensured color_count column")

    df = transform_rank_columns(df)
    logger.log("Transformed rank columns")

    df = transform_price(df)
    logger.log("Transformed price")

    df = transform_quantity_sold(df)
    logger.log("Transformed quantity sold")

    df = transform_discount(df)
    logger.log("Transformed discount")

    df = handle_nulls(df)
    logger.log("Handled null values")

    logger.log_df(df, "After Transformations")

    return df

# 4. Data merging
- Combine all category DataFrames into one
- Ensure schema consistency across all dataset

## (n) - `enforce_schema(df)`
Enforces expected data types across transformed columns.

In [0]:
def enforce_schema(df):
    return (
        df
        .withColumn("rank", expr("try_cast(rank as int)"))
        .withColumn("price_usd", expr("try_cast(price_usd as float)"))
        .withColumn("pct_discount", expr("try_cast(pct_discount as float)"))
        .withColumn("qty_sold", expr("try_cast(qty_sold as int)"))
        .withColumn("color_count", expr("try_cast(color_count as int)"))
    )

# 5. Data Quality Checks
- Identify duplicate records

- Remove or handle duplicates appropriately

- Validate null values in key columns

- Validate data types after transformation

## (o) - `handle_nulls(df)`
Handles missing and null values in selected columns.

In [0]:
def handle_nulls(df):
    if "rank_category" in df.columns:
        df = df.fillna({"rank_category": "Unknown"})
    
    if "rank" in df.columns:
        df = df.fillna({"rank": 0})

    if "pct_discount" in df.columns:
        df = df.fillna({"pct_discount": 0.0})

    return df

## (p) - `remove_duplicates(df)`
Identifies and removes duplicate records.

In [0]:
def remove_duplicates(df):
    dedup_df = df.dropDuplicates()
    logger.log_df(dedup_df, "After Deduplication")
    return dedup_df

# 6. Output
- Save the final cleaned dataset

- Preferred format: CSV (Parquet Optional)

In [0]:
logger = DualLogger()

files = [
    os.path.join(INPUT_PATH, f)
    for f in os.listdir(INPUT_PATH)
    if f.endswith(".csv")
]

if not files:
    raise ValueError("No input files found")

combined_df = load_and_combine(files)

df = apply_transformations(combined_df)

df = enforce_schema(df)
logger.log("Schema enforced using try_cast")
logger.log_df(df, "After Schema Enforcement")

final_df = select_required_columns(df)
logger.log_df(final_df, "Final DataFrame")

final_df = remove_duplicates(final_df)

logger.log("Writing output to target location")

final_df.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(os.path.join(OUTPUT_PATH, "consolidated_data.csv"))

logger.log("Write completed successfully")
logger.close()